# Phase 9 — Contrôle

Ce carnet relance le module, puis il vérifie que la ligne sans choc retrouve les phases 5 et 6, et que chaque choc retrouve son paramètre.


In [1]:

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "src").is_dir():
        PROJECT_ROOT = candidate
        break
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src.config import (
    MARKET_SHOCK_RETURN,
    RECESSION_SHOCK_MONTHS,
    RECESSION_SHOCK_RETURN,
    VOLATILITY_SHOCK_MULTIPLIER,
)
from src.portfolio_construction.build_portfolios import METRICS_FILE, STRATEGY_IDS
from src.portfolio_monitoring.build_monitoring import PATH_FILE
from src.stress_testing.build_stress import (
    AGREEMENT,
    PATHS_FILE,
    RESULTS_FILE,
    SHOCK_IDS,
    build_stress,
)
from src.stress_testing.dashboard import DASHBOARD_FILE, SHOCK_LABELS

written = build_stress()
results = pd.read_csv(RESULTS_FILE)
paths = pd.read_csv(PATHS_FILE)
metrics = pd.read_csv(METRICS_FILE)
wealth = pd.read_csv(PATH_FILE)
assert list(results["strategy"].unique()) == list(STRATEGY_IDS)
assert set(results["shock"]) == set(SHOCK_IDS)
assert results["window_end"].nunique() == 1
assert written["dashboard"] == DASHBOARD_FILE
html = DASHBOARD_FILE.read_text(encoding="utf-8")
assert "Tests de résistance" in html
assert str(results["window_end"].iloc[0]) in html
for strategy in STRATEGY_IDS:
    block = results.loc[results["strategy"].eq(strategy)].set_index("shock")
    official = metrics.loc[metrics["strategy"].eq(strategy)].iloc[0]
    base = block.loc["aucun"]
    for name in ("sharpe_ratio", "maximum_drawdown", "annualized_volatility"):
        assert abs(float(base[name]) - float(official[name])) <= AGREEMENT
    path = wealth.loc[wealth["strategy"].eq(strategy)].sort_values("month")
    assert abs(float(path["cumulative_wealth"].iloc[-1]) - float(base["terminal_wealth"])) <= AGREEMENT
    for name in ("terminal_wealth", "maximum_drawdown", "annualized_volatility"):
        assert abs(float(block.loc["taux", name]) - float(base[name])) <= AGREEMENT
    assert abs(float(block.loc["taux", "sharpe_ratio"]) - float(base["sharpe_ratio"])) > AGREEMENT
    ratio = float(block.loc["volatilite", "annualized_volatility"]) / float(base["annualized_volatility"])
    assert abs(ratio - VOLATILITY_SHOCK_MULTIPLIER) <= AGREEMENT
    trail = paths.loc[paths["strategy"].eq(strategy)].sort_values("month")
    market = float(trail.loc[trail["shock"].eq("marche"), "portfolio_return"].iloc[-1])
    assert abs(market - MARKET_SHOCK_RETURN) <= AGREEMENT
    recession = trail.loc[trail["shock"].eq("recession"), "portfolio_return"].iloc[-RECESSION_SHOCK_MONTHS :]
    assert float(np.max(np.abs(recession.to_numpy(dtype=float) - RECESSION_SHOCK_RETURN))) <= AGREEMENT
    kept = trail.loc[trail["shock"].eq("recession"), "portfolio_return"].iloc[:-RECESSION_SHOCK_MONTHS]
    original = trail.loc[trail["shock"].eq("aucun"), "portfolio_return"].iloc[:-RECESSION_SHOCK_MONTHS]
    assert float(np.max(np.abs(kept.to_numpy(dtype=float) - original.to_numpy(dtype=float)))) <= AGREEMENT
assert list(SHOCK_LABELS) == list(SHOCK_IDS)
print("Résultat : PASS")


Résultat : PASS
